In [1]:
import time
from pathlib import Path
import requests
import pandas as pd
import numpy as np
from src.utils import SimpleTimeFormater
from src.app.db_querriees import db_connector
from secrets_local import DB_URL

PREDICTOR_URL = "http://localhost:8000/predict"


def send_predict_requests(
    df: pd.DataFrame,
    url: str,
    pause_seconds: float = 0.05,
    drop_columns: list[str] | None = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Отправляет строки DataFrame как POST-запросы на FastAPI /predict.

    Parameters
    ----------
    df:
        DataFrame с признаками.
    url:
        Адрес endpoint".
    pause_seconds:
        Пауза между запросами.
    drop_columns:
        Колонки, которые нужно убрать перед отправкой.
    """
    timeout_seconds = 1.0
    data = df.copy()
    if drop_columns:
        data = df.drop(columns=drop_columns, errors="ignore")

    for i, idx in enumerate(data.index):
        payload = data.loc[idx].to_dict()
        start = time.time()

        try:
            response = requests.post(
                url,
                json=payload,
                timeout=timeout_seconds,
            )
            response = response.json()
        except Exception as e:
            response = {"Attrition": -200, "Error": e, "Your_case_id": -9999}
        finally:
            response["time"] = time.time() - start
            df.loc[idx, "Your_case_id"] = int(response["Your_case_id"])

        if verbose:
            if response["Attrition"] == 0:
                response["Attrition"] = "No"
            elif response["Attrition"] == 1:
                response["Attrition"] = "Yes"
            elif response["Attrition"] == -100:
                response["Attrition"] = "Service in test mode"
            elif response["Attrition"] == -200:
                response["Attrition"] = f"No connection, {response["Error"]}"
            print(
                f"Row {i + 1}: {response["Attrition"]}, "
                f"responce time: {SimpleTimeFormater(response["time"])}"
            )

        if pause_seconds > 0 and i < len(data):
            time.sleep(pause_seconds)

In [2]:
no_drift = pd.read_csv(
    Path.cwd().parent / "data" / "interim" / "no_drift.csv",
    index_col="Unnamed: 0",
).reset_index()

concept_drift = pd.read_csv(
    Path.cwd().parent / "data" / "interim" / "concept_drift.csv",
    index_col="Unnamed: 0",
).reset_index()

synt_1 = pd.read_csv(
    Path.cwd().parent / "data" / "interim" / "synt_1.csv",
    index_col="Unnamed: 0",
).reset_index()

target_drift = pd.read_csv(
    Path.cwd().parent / "data" / "interim" / "target_drift.csv",
    index_col="Unnamed: 0",
).reset_index()

no_drift = no_drift.iloc[:1470]

synt_2 = pd.read_csv(
    Path.cwd().parent / "data" / "interim" / "synt_2.csv",
    index_col="Unnamed: 0",
).reset_index()

data_drift = pd.read_csv(
    Path.cwd().parent / "data" / "interim" / "datadrift.csv",
    index_col="Unnamed: 0",
).reset_index()

synt_3 = pd.read_csv(
    Path.cwd().parent / "data" / "interim" / "synt_3.csv",
    index_col="Unnamed: 0",
).reset_index()


posts_per_second = 1000

db_posts_per_second = 1000

time_to_wait = 0
verbose = True

db_pause_in_seconds = 1 / db_posts_per_second
pause_in_seconds = 1 / posts_per_second

### Данные без дрифта

In [3]:
send_predict_requests(
    df=no_drift, url=PREDICTOR_URL, pause_seconds=0, verbose=verbose
)

print("done")

Row 1: Yes, responce time: 71ms
Row 2: No, responce time: 50ms
Row 3: Yes, responce time: 101ms
Row 4: No, responce time: 135ms
Row 5: No, responce time: 63ms
Row 6: No, responce time: 54ms
Row 7: No, responce time: 46ms
Row 8: Yes, responce time: 59ms
Row 9: No, responce time: 67ms
Row 10: No, responce time: 47ms
Row 11: No, responce time: 47ms
Row 12: Yes, responce time: 50ms
Row 13: No, responce time: 46ms
Row 14: Yes, responce time: 50ms
Row 15: No, responce time: 47ms
Row 16: No, responce time: 48ms
Row 17: No, responce time: 50ms
Row 18: No, responce time: 50ms
Row 19: No, responce time: 66ms
Row 20: No, responce time: 49ms
Row 21: Yes, responce time: 56ms
Row 22: No, responce time: 46ms
Row 23: No, responce time: 50ms
Row 24: No, responce time: 50ms
Row 25: Yes, responce time: 50ms
Row 26: No, responce time: 50ms
Row 27: Yes, responce time: 52ms
Row 28: No, responce time: 48ms
Row 29: Yes, responce time: 62ms
Row 30: No, responce time: 56ms
Row 31: No, responce time: 53ms
Row 32

### Запросы с concept drift

In [4]:
send_predict_requests(
    df=concept_drift,
    url=PREDICTOR_URL,
    pause_seconds=pause_in_seconds,
    verbose=verbose,
)

print("done")

Row 1: No, responce time: 64ms
Row 2: Yes, responce time: 52ms
Row 3: No, responce time: 45ms
Row 4: Yes, responce time: 56ms
Row 5: No, responce time: 60ms
Row 6: No, responce time: 55ms
Row 7: No, responce time: 46ms
Row 8: Yes, responce time: 46ms
Row 9: No, responce time: 61ms
Row 10: No, responce time: 45ms
Row 11: Yes, responce time: 46ms
Row 12: No, responce time: 50ms
Row 13: No, responce time: 48ms
Row 14: Yes, responce time: 47ms
Row 15: No, responce time: 47ms
Row 16: No, responce time: 52ms
Row 17: No, responce time: 48ms
Row 18: Yes, responce time: 49ms
Row 19: No, responce time: 48ms
Row 20: Yes, responce time: 51ms
Row 21: No, responce time: 51ms
Row 22: No, responce time: 55ms
Row 23: No, responce time: 48ms
Row 24: No, responce time: 49ms
Row 25: No, responce time: 54ms
Row 26: No, responce time: 53ms
Row 27: No, responce time: 49ms
Row 28: No, responce time: 45ms
Row 29: No, responce time: 47ms
Row 30: Yes, responce time: 56ms
Row 31: No, responce time: 48ms
Row 32: N

### Для детекции нужны реальные метки

Они поступают

In [5]:
postgres_attr = db_connector(DB_URL)


def pull_gt_labels(df):
    for jj in range(len(df)):
        row_gt = {
            "row_id": int(df.iloc[jj]["Your_case_id"]),
            "GT_Attrition": 1 if df.iloc[jj]["Attrition"] == "Yes" else 0,
        }
        postgres_attr.insert_gt_labels(row_gt)
        time.sleep(db_pause_in_seconds)


pull_gt_labels(no_drift)
pull_gt_labels(concept_drift)

print("done")

done


### Срабатывает детекция дрифта и модель переобучается (подождать)

In [6]:
time.sleep(time_to_wait)

### Далее идут хорошие данные

In [7]:
send_predict_requests(
    df=synt_1,
    url=PREDICTOR_URL,
    pause_seconds=pause_in_seconds,
    verbose=verbose,
)

print("done")

Row 1: No, responce time: 63ms
Row 2: Yes, responce time: 50ms
Row 3: No, responce time: 46ms
Row 4: No, responce time: 45ms
Row 5: No, responce time: 46ms
Row 6: Yes, responce time: 49ms
Row 7: Yes, responce time: 44ms
Row 8: No, responce time: 47ms
Row 9: No, responce time: 49ms
Row 10: Yes, responce time: 46ms
Row 11: Yes, responce time: 46ms
Row 12: Yes, responce time: 46ms
Row 13: No, responce time: 56ms
Row 14: No, responce time: 45ms
Row 15: No, responce time: 43ms
Row 16: No, responce time: 45ms
Row 17: Yes, responce time: 44ms
Row 18: No, responce time: 44ms
Row 19: Yes, responce time: 53ms
Row 20: No, responce time: 46ms
Row 21: No, responce time: 44ms
Row 22: No, responce time: 46ms
Row 23: No, responce time: 46ms
Row 24: Yes, responce time: 45ms
Row 25: No, responce time: 45ms
Row 26: No, responce time: 47ms
Row 27: No, responce time: 46ms
Row 28: No, responce time: 45ms
Row 29: No, responce time: 45ms
Row 30: Yes, responce time: 49ms
Row 31: No, responce time: 45ms
Row 32:

### Идут данные с target drift

In [8]:
send_predict_requests(
    df=target_drift,
    url=PREDICTOR_URL,
    pause_seconds=pause_in_seconds,
    verbose=verbose,
)

print("done")

Row 1: Yes, responce time: 56ms
Row 2: No, responce time: 46ms
Row 3: Yes, responce time: 47ms
Row 4: No, responce time: 47ms
Row 5: Yes, responce time: 53ms
Row 6: No, responce time: 65ms
Row 7: No, responce time: 46ms
Row 8: Yes, responce time: 47ms
Row 9: No, responce time: 45ms
Row 10: No, responce time: 45ms
Row 11: No, responce time: 52ms
Row 12: No, responce time: 54ms
Row 13: Yes, responce time: 44ms
Row 14: No, responce time: 46ms
Row 15: Yes, responce time: 45ms
Row 16: No, responce time: 45ms
Row 17: No, responce time: 46ms
Row 18: Yes, responce time: 45ms
Row 19: Yes, responce time: 45ms
Row 20: No, responce time: 44ms
Row 21: No, responce time: 45ms
Row 22: Yes, responce time: 46ms
Row 23: No, responce time: 45ms
Row 24: No, responce time: 45ms
Row 25: Yes, responce time: 48ms
Row 26: No, responce time: 45ms
Row 27: No, responce time: 46ms
Row 28: Yes, responce time: 46ms
Row 29: Yes, responce time: 52ms
Row 30: No, responce time: 50ms
Row 31: No, responce time: 50ms
Row 3

### Им тоже требуются метки для детекции

Они поступают

In [9]:
pull_gt_labels(synt_1)
pull_gt_labels(target_drift)

print("done")

done


### Срабатывает детекция дрифта и модель переобучается (подождать)

In [10]:
time.sleep(time_to_wait)

### Идут хорошие данные

In [11]:
send_predict_requests(
    df=synt_2,
    url=PREDICTOR_URL,
    pause_seconds=pause_in_seconds,
    verbose=verbose,
)

print("done")

Row 1: No, responce time: 52ms
Row 2: No, responce time: 42ms
Row 3: No, responce time: 45ms
Row 4: Yes, responce time: 45ms
Row 5: Yes, responce time: 44ms
Row 6: No, responce time: 44ms
Row 7: No, responce time: 46ms
Row 8: No, responce time: 45ms
Row 9: No, responce time: 48ms
Row 10: No, responce time: 56ms
Row 11: Yes, responce time: 48ms
Row 12: No, responce time: 43ms
Row 13: Yes, responce time: 45ms
Row 14: No, responce time: 47ms
Row 15: No, responce time: 52ms
Row 16: No, responce time: 49ms
Row 17: No, responce time: 46ms
Row 18: No, responce time: 49ms
Row 19: Yes, responce time: 45ms
Row 20: No, responce time: 45ms
Row 21: No, responce time: 48ms
Row 22: Yes, responce time: 48ms
Row 23: Yes, responce time: 48ms
Row 24: No, responce time: 47ms
Row 25: Yes, responce time: 45ms
Row 26: No, responce time: 47ms
Row 27: No, responce time: 48ms
Row 28: No, responce time: 45ms
Row 29: No, responce time: 45ms
Row 30: No, responce time: 44ms
Row 31: No, responce time: 45ms
Row 32: Y

### Идут данные с data drift

для детекции метки не нужны

In [12]:
send_predict_requests(
    df=data_drift,
    url=PREDICTOR_URL,
    pause_seconds=pause_in_seconds,
    verbose=verbose,
)

print("done")

Row 1: No, responce time: 47ms
Row 2: Yes, responce time: 47ms
Row 3: Yes, responce time: 46ms
Row 4: Yes, responce time: 47ms
Row 5: Yes, responce time: 46ms
Row 6: Yes, responce time: 48ms
Row 7: No, responce time: 45ms
Row 8: Yes, responce time: 45ms
Row 9: Yes, responce time: 45ms
Row 10: No, responce time: 49ms
Row 11: Yes, responce time: 48ms
Row 12: Yes, responce time: 46ms
Row 13: No, responce time: 43ms
Row 14: Yes, responce time: 47ms
Row 15: Yes, responce time: 46ms
Row 16: No, responce time: 49ms
Row 17: Yes, responce time: 46ms
Row 18: Yes, responce time: 49ms
Row 19: No, responce time: 44ms
Row 20: Yes, responce time: 49ms
Row 21: No, responce time: 44ms
Row 22: Yes, responce time: 46ms
Row 23: No, responce time: 48ms
Row 24: No, responce time: 48ms
Row 25: Yes, responce time: 47ms
Row 26: Yes, responce time: 50ms
Row 27: No, responce time: 48ms
Row 28: Yes, responce time: 45ms
Row 29: No, responce time: 45ms
Row 30: Yes, responce time: 44ms
Row 31: Yes, responce time: 47

### Поступают метки для data drift

In [13]:
pull_gt_labels(data_drift)
pull_gt_labels(synt_2)

print("done")

done


### Срабатывает детекция, что данные появлились и модель переобучается (подождать)

In [14]:
time.sleep(time_to_wait)

### Идут хорошие данные

In [15]:
send_predict_requests(
    df=synt_3,
    url=PREDICTOR_URL,
    pause_seconds=pause_in_seconds,
    verbose=verbose,
)

print("done")

Row 1: No, responce time: 50ms
Row 2: Yes, responce time: 44ms
Row 3: Yes, responce time: 48ms
Row 4: Yes, responce time: 52ms
Row 5: No, responce time: 45ms
Row 6: No, responce time: 45ms
Row 7: Yes, responce time: 50ms
Row 8: No, responce time: 48ms
Row 9: No, responce time: 51ms
Row 10: No, responce time: 46ms
Row 11: No, responce time: 43ms
Row 12: No, responce time: 47ms
Row 13: Yes, responce time: 44ms
Row 14: Yes, responce time: 51ms
Row 15: No, responce time: 47ms
Row 16: No, responce time: 48ms
Row 17: No, responce time: 48ms
Row 18: Yes, responce time: 63ms
Row 19: No, responce time: 42ms
Row 20: No, responce time: 47ms
Row 21: No, responce time: 44ms
Row 22: No, responce time: 45ms
Row 23: Yes, responce time: 45ms
Row 24: No, responce time: 48ms
Row 25: No, responce time: 44ms
Row 26: Yes, responce time: 44ms
Row 27: No, responce time: 44ms
Row 28: No, responce time: 45ms
Row 29: Yes, responce time: 45ms
Row 30: No, responce time: 50ms
Row 31: No, responce time: 44ms
Row 32: